# Artery-and-vein benchmark — analysis

The generated pages hold the facts: [how it is run](../docs/benchmarks/av-docs.md) and
[what came out](../docs/benchmarks/av-results.md). This notebook is where the judgement goes.

The benchmark's question is **model against the vessels an expert drew**, so the model is the
primary index everywhere below, and the dataset and the structure sit inside it. Three maps are
scored: the **arteries**, the **veins**, and the **vessels** they make together — the last derived
as the union of the other two, identically for the model and for the annotator, so that every
model's vessel score means the same thing.

Every measurement is in the **native frame** — the full-resolution square the store built, which is
where the annotator drew — so a pixel here is a pixel of the original photograph.

Run it after `python -m benchmarks --benchmark av`.

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image


def repository() -> Path:
    """The repository, found rather than assumed.

    A notebook is opened from wherever its reader happens to be — the repository root, the
    notebooks directory, an editor's workspace — and a relative path that is right in one of those
    is wrong in the others.
    """
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").is_dir():
            return candidate
    raise RuntimeError(
        "this notebook has to be run from inside the fundus-atlas repository; "
        f"nothing above {Path.cwd()} looks like it"
    )


ROOT = repository()
sys.path.insert(0, str(ROOT / "src"))
RESULTS = ROOT / "results" / "av"
STORE = ROOT / ".atlas_data"
KEPT_MASKS = ROOT / ".atlas_runs" / "av"

# The run's own measurements and marks, reused here rather than implemented a second time.
from benchmarks import av as benchmark  # noqa: E402
from benchmarks import contamination  # noqa: E402
from benchmarks.metrics import av as measurements  # noqa: E402

STRUCTURES = ["artery", "vein", "vessels"]
Image.MAX_IMAGE_PIXELS = None


def per_image() -> pd.DataFrame:
    """Every model's answer about every photograph, one row per annotation it was scored against."""
    frames = []
    for path in sorted(RESULTS.glob("*/*.csv")):
        frame = pd.read_csv(path)
        frame["model"] = path.parent.name
        frame["dataset"] = path.stem
        frames.append(frame)
    if not frames:
        raise FileNotFoundError(
            f"no results in {RESULTS} — run `python -m benchmarks --benchmark av` first"
        )
    return pd.concat(frames, ignore_index=True)


def summaries() -> pd.DataFrame:
    """What each run recorded about itself: how much it covered, and how long it took."""
    rows = []
    for path in sorted(RESULTS.glob("*/*.json")):
        record = json.loads(path.read_text())
        kept = record["summary"]
        rows.append(
            {
                "model": record["model"],
                "dataset": record["dataset"],
                "photographs": kept.get("photographs"),
                "segmentations": kept.get("segmentations"),
                "failed": kept.get("failed"),
                "processed": kept.get("processed"),
                "total": kept.get("total"),
                "complete": kept.get("complete"),
                "excluded": "; ".join(
                    f"{count} {reason}" for reason, count in sorted(kept.get("excluded", {}).items())
                ),
                "seconds_per_photograph": kept.get("seconds_per_photograph"),
                "device": kept.get("device"),
            }
        )
    return pd.DataFrame(rows).set_index(["model", "dataset"]).sort_index()


def manifest(dataset: str) -> pd.DataFrame:
    """One dataset's manifest, for the facts about a photograph the results do not carry."""
    return pd.read_csv(STORE / dataset / "manifest.csv", dtype=str).set_index("key")


def drawn(dataset: str, key: str, structure: str) -> np.ndarray:
    """What the annotator drew, in the native frame."""
    with Image.open(STORE / dataset / "native" / structure / f"{key}.png") as mask:
        return np.asarray(mask) > 0


def said(model: str, dataset: str, key: str, structure: str) -> np.ndarray:
    """What a model drew, read back from the masks the run kept.

    These are the files the biomarker benchmark will read: one 1-bit PNG per structure, in the
    native frame, under `.atlas_runs/av/<model>/<dataset>/`.
    """
    with Image.open(KEPT_MASKS / model / dataset / f"{key}-{structure}.png") as mask:
        return np.asarray(mask) > 0


rows = per_image()
overview = summaries()
MODELS = sorted(rows["model"].unique())
DATASETS = sorted(rows["dataset"].unique())
GRADED = rows[rows["outcome"] == "graded"].copy()
for structure in STRUCTURES:
    for metric in ("dice", "cldice", "betti"):
        column = f"{structure}_{metric}"
        if column in GRADED:
            GRADED[column] = pd.to_numeric(GRADED[column], errors="coerce")

print(f"{len(rows):,} scored segmentations · {len(MODELS)} models · {len(DATASETS)} datasets")
print("models:", ", ".join(MODELS))
print("datasets:", ", ".join(DATASETS))
if not overview["complete"].all():
    print("\nPARTIAL — these pairs are not finished, and every figure below is of what exists:")
    print(overview.loc[~overview["complete"], ["processed", "total"]].to_string())

## 1. Coverage

A model that produced no segmentation has not got one wrong. None of these networks has a way of
declining, so anything other than `graded` is a failure with a reason attached, and it is counted
separately from the photographs the benchmark itself excluded.

The counts are **segmentations**: one per photograph here, because no dataset built for this
benchmark keeps its annotators apart (section 5).

In [ ]:
coverage = (
    rows.pivot_table(index=["model", "dataset"], columns="outcome", values="key", aggfunc="count")
    .fillna(0)
    .astype(int)
)
coverage.join(overview[["processed", "total", "complete", "excluded"]])

In [ ]:
failures = rows[rows["outcome"] != "graded"]
if failures.empty:
    print("every photograph was segmented by every model")
else:
    print(failures.groupby(["model", "dataset"])["note"].value_counts().to_string())

### 1.1 What the benchmark itself set aside

`excluded` above is **ours**, not the models': a photograph below the size floor, one with no
artery/vein annotation to score against, or one a recorded finding condemns. It is never added to a
model's failures — one says the benchmark would not ask, the other that the model could not answer.

In [ ]:
# The exclusions are the benchmark's, so every model that ran on a dataset must report the same
# ones. Read from whichever models ran — a model still to be run has no row, which is not a
# disagreement — and say so where they differ, because then one of the runs is reading a store the
# other did not.
aside = []
for dataset in DATASETS:
    ran = overview.xs(dataset, level="dataset")
    agreed = ran[["total", "excluded"]].drop_duplicates()
    aside.append(
        {
            "dataset": dataset,
            "scored": int(ran["total"].iloc[0]),
            "set aside": ran["excluded"].iloc[0] or "none",
            "models run": len(ran),
            "models agree": len(agreed) == 1,
        }
    )
pd.DataFrame(aside).set_index("dataset")

## 2. Overlap, briefly

**Dice** is the familiar number: how much of the annotated area the model found, 1 being perfect
agreement and 0 no overlap at all. It is the least informative number here, because a vessel map
can be right about area and wrong about everything a measurement is taken from — which is what
section 3 is for.

Read the **vessels** column as a different question from the other two: it says whether the model
found the vessels at all, while `artery` and `vein` say whether it named them correctly.

**clDice and the Betti matching error are tabled and plotted beneath each Dice figure below**, in
the same shape, so a model's three numbers can be read together. What their differences *mean* is
section 3; here they are put side by side and nothing is concluded from the set.

The third one counts rather than scores. **Betti matching error** is how many topological features
— connected components and loops — of either map have no counterpart in the other, so **0 is
perfect and it has no ceiling**: a model that shatters a vessel network into a thousand fragments
scores in the thousands. It is the only number here that grows with the size of the photograph as
well as with the mistake, which is why it is never pooled across datasets without saying so.

In [ ]:
pooled = GRADED.groupby("model")[[f"{s}_dice" for s in STRUCTURES]].agg(
    ["mean", "median", "std", "count"]
)
pooled.round(3)

In [ ]:
# The same table for clDice, in the same shape, so a model's two numbers can be read against each
# other without scrolling. What the difference between them means is section 3's question; here
# they are only put side by side.
GRADED.groupby("model")[[f"{s}_cldice" for s in STRUCTURES]].agg(
    ["mean", "median", "std", "count"]
).round(3)

In [ ]:
# And the same table for the Betti matching error. The median matters more than the mean here: a
# count with no ceiling has a tail that drags a mean wherever the worst photograph went.
GRADED.groupby("model")[[f"{s}_betti" for s in STRUCTURES]].agg(
    ["mean", "median", "std", "count"]
).round(1)

### 2.1 The same numbers, per dataset

Pooling hides the datasets, and these are not samples of one thing: different cameras, different
populations, different hands drawing the vessels, and native frames from under a thousand pixels to
over three thousand.

Dice first, clDice directly beneath it on the same colour scale, then the Betti matching error on a
scale of its own — it is a count, not a fraction, and putting it on 0 to 1 would say nothing. A
cell that changes colour between the first two is a model and a dataset where width and network
disagree; the third says how many features that cost.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for axis, structure in zip(axes, STRUCTURES):
    table = GRADED.pivot_table(
        index="model", columns="dataset", values=f"{structure}_dice", aggfunc="mean"
    )
    drawn_table = axis.imshow(table.values, cmap="viridis", vmin=0.2, vmax=0.9, aspect="auto")
    axis.set_xticks(range(len(table.columns)), table.columns, rotation=30, ha="right")
    axis.set_yticks(range(len(table.index)), table.index)
    axis.set_title(f"{structure} — mean Dice")
    for y in range(table.shape[0]):
        for x in range(table.shape[1]):
            value = table.values[y, x]
            if not np.isnan(value):
                axis.text(
                    x, y, f"{value:.3f}", ha="center", va="center",
                    color="white" if value < 0.6 else "black", fontsize=9,
                )
    fig.colorbar(drawn_table, ax=axis, shrink=0.85)
fig.suptitle("Overlap with what the annotator drew, by model and dataset")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for axis, structure in zip(axes, STRUCTURES):
    table = GRADED.pivot_table(
        index="model", columns="dataset", values=f"{structure}_cldice", aggfunc="mean"
    )
    drawn_table = axis.imshow(table.values, cmap="viridis", vmin=0.2, vmax=0.9, aspect="auto")
    axis.set_xticks(range(len(table.columns)), table.columns, rotation=30, ha="right")
    axis.set_yticks(range(len(table.index)), table.index)
    axis.set_title(f"{structure} — mean clDice")
    for y in range(table.shape[0]):
        for x in range(table.shape[1]):
            value = table.values[y, x]
            if not np.isnan(value):
                axis.text(
                    x, y, f"{value:.3f}", ha="center", va="center",
                    color="white" if value < 0.6 else "black", fontsize=9,
                )
    fig.colorbar(drawn_table, ax=axis, shrink=0.85)
# The same scale as the Dice heatmaps above, deliberately: read one against the other cell by cell
# and the cells that change colour are where width and network disagree.
fig.suptitle("Connectedness of what the annotator drew, by model and dataset")
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
for axis, structure in zip(axes, STRUCTURES):
    table = GRADED.pivot_table(
        index="model", columns="dataset", values=f"{structure}_betti", aggfunc="median"
    )
    # Its own scale, and a reversed one: unlike Dice and clDice, low is good and there is no 1.
    drawn_table = axis.imshow(table.values, cmap="viridis_r", aspect="auto")
    axis.set_xticks(range(len(table.columns)), table.columns, rotation=30, ha="right")
    axis.set_yticks(range(len(table.index)), table.index)
    axis.set_title(f"{structure} — median Betti matching error")
    for y in range(table.shape[0]):
        for x in range(table.shape[1]):
            value = table.values[y, x]
            if not np.isnan(value):
                axis.text(x, y, f"{value:.0f}", ha="center", va="center", color="black", fontsize=9)
    fig.colorbar(drawn_table, ax=axis, shrink=0.85)
fig.suptitle("Unmatched topological features, by model and dataset — lower is better")
fig.tight_layout()

### 2.2 The distribution behind each mean

A mean Dice of 0.70 can be every photograph at 0.70, or most at 0.80 with a tail of photographs the
model lost altogether. The tail is what a study feels, and it is invisible in a table of means. The
clDice distributions follow, drawn the same way: a model whose overlap has a long tail and whose
connectedness does not lost width on those photographs rather than the vessel.

The Betti distributions come last, on a logarithmic scale — a count with no ceiling has a tail that
a linear axis turns into one line at the bottom and a few dots at the top.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, structure in zip(axes, STRUCTURES):
    series = [(m, GRADED.loc[GRADED["model"] == m, f"{structure}_dice"].dropna()) for m in MODELS]
    present = [(m, values) for m, values in series if len(values)]
    axis.boxplot(
        [values for _, values in present],
        tick_labels=[m for m, _ in present],
        showfliers=True,
        flierprops={"markersize": 2, "alpha": 0.3},
    )
    axis.set_title(f"{structure} Dice, every photograph")
    axis.set_ylim(0, 1.02)
    axis.tick_params(axis="x", rotation=30)
    axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, structure in zip(axes, STRUCTURES):
    series = [(m, GRADED.loc[GRADED["model"] == m, f"{structure}_cldice"].dropna()) for m in MODELS]
    present = [(m, values) for m, values in series if len(values)]
    axis.boxplot(
        [values for _, values in present],
        tick_labels=[m for m, _ in present],
        showfliers=True,
        flierprops={"markersize": 2, "alpha": 0.3},
    )
    axis.set_title(f"{structure} clDice, every photograph")
    axis.set_ylim(0, 1.02)
    axis.tick_params(axis="x", rotation=30)
    axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for axis, structure in zip(axes, STRUCTURES):
    series = [(m, GRADED.loc[GRADED["model"] == m, f"{structure}_betti"].dropna()) for m in MODELS]
    present = [(m, values) for m, values in series if len(values)]
    axis.boxplot(
        [values.clip(lower=0.5) for _, values in present],
        tick_labels=[m for m, _ in present],
        showfliers=True,
        flierprops={"markersize": 2, "alpha": 0.3},
    )
    # Logarithmic, because a count with no ceiling puts every model on one line otherwise. A
    # photograph with no unmatched feature at all is drawn at the floor rather than dropped.
    axis.set_yscale("log")
    axis.set_title(f"{structure} Betti matching error, every photograph")
    axis.tick_params(axis="x", rotation=30)
    axis.grid(axis="y", alpha=0.3)
fig.tight_layout()

In [ ]:
# How often a model is not merely imprecise but somewhere else entirely.
lost = {
    structure: {
        model: float(
            (GRADED.loc[GRADED["model"] == model, f"{structure}_dice"].dropna() < 0.5).mean()
        )
        for model in MODELS
    }
    for structure in STRUCTURES
}
pd.DataFrame(lost).rename(columns=lambda name: f"share of {name} maps under Dice 0.5").round(4)

## 3. Connectedness, read beside overlap

**clDice** asks a different question: what share of the model's *centreline* falls inside the
expert's vessels, and what share of the expert's centreline falls inside the model's, combined as a
harmonic mean. It is a question about the network rather than about the area.

The pair separates two failures overlap alone confuses. A vessel drawn three times too wide around
the same centre scores 0.50 Dice and 0.97 clDice — wrong about width, right about the network. The
same vessel thickened to one side scores the same 0.50 Dice and 0.03 clDice, its centreline no
longer inside the expert's vessel at all. A break in a vessel costs both alike.

In [ ]:
GRADED.groupby("model")[[f"{s}_cldice" for s in STRUCTURES]].agg(
    ["mean", "median", "count"]
).round(3)

### 3.1 Dice against clDice, one photograph at a time

The means are two numbers; **the shape of this scatter is the finding**. The diagonal is where the
two agree. Above it — clDice higher than Dice — the model traced the right network at the wrong
width, which a calibre measurement will feel and a connectivity one will not. Below it, the model
covered the area and lost the network.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(3.4 * len(MODELS), 3.6), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for axis, model in zip(axes, MODELS):
    mine = GRADED[GRADED["model"] == model]
    for dataset in DATASETS:
        theirs = mine[mine["dataset"] == dataset]
        axis.scatter(
            theirs["vessels_dice"], theirs["vessels_cldice"], s=6, alpha=0.45, label=dataset
        )
    axis.plot([0, 1], [0, 1], color="black", linewidth=0.8, linestyle="--")
    axis.set_title(model, fontsize=10)
    axis.set_xlabel("vessels Dice")
    axis.set_xlim(0, 1)
    axis.set_ylim(0, 1)
    axis.grid(alpha=0.3)
axes[0].set_ylabel("vessels clDice")
axes[-1].legend(fontsize=7, markerscale=1.5, loc="lower right")
fig.suptitle("Overlap against connectedness — one point per photograph")
fig.tight_layout()

In [ ]:
# Which side of the diagonal each model tends to sit on, and by how much.
gap = GRADED.assign(gap=GRADED["vessels_cldice"] - GRADED["vessels_dice"])
pd.DataFrame(
    {
        "median clDice − Dice": gap.groupby("model")["gap"].median(),
        "share traced-not-covered (gap > +0.05)": gap.groupby("model")["gap"].apply(
            lambda values: float((values > 0.05).mean())
        ),
        "share covered-not-traced (gap < −0.05)": gap.groupby("model")["gap"].apply(
            lambda values: float((values < -0.05).mean())
        ),
    }
).round(3)

### 3.2 Dice against Betti matching error, one photograph at a time

Dice and clDice both score pixels; the Betti matching error **counts unmatched features** instead.
There is no diagonal: one axis is a fraction, the other a count, and they point opposite ways — a
good photograph sits at the **bottom right**, high overlap and few unmatched features.

**The shape is the finding again.** A slope down and to the right would mean overlap and topology
agree; a cloud would mean a model's Dice tells you almost nothing about whether its vessel network
is in one piece. Colour is the dataset, because a 3,269-pixel HRF frame has more features to leave
unmatched than a 1,062-pixel AVRDB one whatever the model did.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(3.4 * len(MODELS), 3.6), sharex=True, sharey=True)
axes = np.atleast_1d(axes)
for axis, model in zip(axes, MODELS):
    mine = GRADED[GRADED["model"] == model]
    for dataset in DATASETS:
        theirs = mine[mine["dataset"] == dataset]
        axis.scatter(
            theirs["vessels_dice"],
            theirs["vessels_betti"].clip(lower=0.5),
            s=6,
            alpha=0.45,
            label=dataset,
        )
    axis.set_title(model, fontsize=10)
    axis.set_xlabel("vessels Dice")
    axis.set_xlim(0, 1)
    # Logarithmic, for the same reason as the box plots: a count with no ceiling otherwise
    # puts every photograph on one line at the bottom. A photograph with no unmatched
    # feature at all is drawn at the floor rather than dropped.
    axis.set_yscale("log")
    axis.grid(alpha=0.3)
axes[0].set_ylabel("vessels Betti matching error")
axes[-1].legend(fontsize=7, markerscale=1.5, loc="upper right")
fig.suptitle("Overlap against unmatched topology — one point per photograph")
fig.tight_layout()

In [ ]:
# How far overlap and topology move together. Spearman, because Betti's tail would dominate a
# Pearson coefficient. Pooled across datasets first, then per dataset — a pooled number mixes
# frames of different sizes, which is exactly what the colour in the scatter is there to show.
def spearman(frame: pd.DataFrame) -> float:
    pair = frame[["vessels_dice", "vessels_betti"]].dropna()
    if len(pair) < 3:
        return float("nan")
    return float(pair["vessels_dice"].corr(pair["vessels_betti"], method="spearman"))


across_datasets = pd.Series({model: spearman(GRADED[GRADED["model"] == model]) for model in MODELS})
by_dataset = pd.DataFrame(
    {
        dataset: {
            model: spearman(GRADED[(GRADED["model"] == model) & (GRADED["dataset"] == dataset)])
            for model in MODELS
        }
        for dataset in DATASETS
    }
)
together = pd.concat([across_datasets.rename("pooled"), by_dataset], axis=1)
together.round(3)

## 4. Arteries against veins — the question this benchmark exists for

A vessel map alone cannot be got wrong in the way an artery/vein map can. Where `vessels` is high
and `artery`/`vein` are low, the model **finds the vessels and cannot tell them apart** — a
different failure from missing them, and one that no single overlap number distinguishes.

In [ ]:
naming = pd.DataFrame(
    {
        "vessels Dice": GRADED.groupby("model")["vessels_dice"].mean(),
        "artery Dice": GRADED.groupby("model")["artery_dice"].mean(),
        "vein Dice": GRADED.groupby("model")["vein_dice"].mean(),
    }
)
naming["cost of naming them"] = naming["vessels Dice"] - naming[["artery Dice", "vein Dice"]].mean(
    axis=1
)
naming.sort_values("cost of naming them").round(3)

### 4.1 How much of the disagreement is swapping

The share of the expert's artery pixels a model called vein, and the reverse. A model that swaps a
whole branch is not making small errors — an arteriovenous ratio computed from it is wrong in a way
no overlap number reveals.

This reads **the masks the run kept** under `.atlas_runs/av/`, which is the same path the biomarker
benchmark will read them by, so it doubles as a check that they are there and in the right frame.
It is a **sample** — the photographs named below, per dataset — because it opens four
full-resolution masks per photograph per model. It covers only the datasets that annotate the two
classes: a dataset that traced vessels without separating them, as FIVES does, has nothing to say
about swapping and is absent from this section and the next.

In [ ]:
SAMPLE = 12
sampled = {
    dataset: sorted(GRADED.loc[GRADED["dataset"] == dataset, "key"].unique())[:SAMPLE]
    for dataset in DATASETS
}

# Which datasets annotate the two classes at all. A dataset that traced vessels and never separated
# arteries from veins — FIVES — has nothing to say about swapping or crossings, and asking it for an
# artery layer is a missing file rather than an answer. It stays in every vessel figure.
CLASSED = {
    dataset: keys
    for dataset, keys in sampled.items()
    if (STORE / dataset / "native" / "artery").is_dir()
}
print({dataset: len(keys) for dataset, keys in sampled.items()})
print("annotating the classes:", ", ".join(CLASSED) or "none")

swaps = []
for model in MODELS:
    for dataset, keys in CLASSED.items():
        for key in keys:
            try:
                truth_artery, truth_vein = drawn(dataset, key, "artery"), drawn(dataset, key, "vein")
                model_artery = said(model, dataset, key, "artery")
                model_vein = said(model, dataset, key, "vein")
            except FileNotFoundError:
                continue
            swaps.append(
                {
                    "model": model,
                    "dataset": dataset,
                    "key": key,
                    "artery called vein": float((truth_artery & model_vein).sum())
                    / max(int(truth_artery.sum()), 1),
                    "vein called artery": float((truth_vein & model_artery).sum())
                    / max(int(truth_vein.sum()), 1),
                    "artery found at all": float((truth_artery & (model_artery | model_vein)).sum())
                    / max(int(truth_artery.sum()), 1),
                }
            )
swapped = pd.DataFrame(swaps)
swapped.groupby("model")[
    ["artery called vein", "vein called artery", "artery found at all"]
].mean().round(3)

In [ ]:
fig, axis = plt.subplots(figsize=(11, 3.6))
table = swapped.pivot_table(index="model", columns="dataset", values="artery called vein")
drawn_table = axis.imshow(table.values, cmap="magma_r", vmin=0, vmax=0.5, aspect="auto")
axis.set_xticks(range(len(table.columns)), table.columns, rotation=30, ha="right")
axis.set_yticks(range(len(table.index)), table.index)
for y in range(table.shape[0]):
    for x in range(table.shape[1]):
        value = table.values[y, x]
        if not np.isnan(value):
            axis.text(x, y, f"{value:.2f}", ha="center", va="center",
                      color="white" if value > 0.25 else "black", fontsize=9)
axis.set_title(f"Share of the expert's artery pixels called vein — {SAMPLE} photographs per dataset")
fig.colorbar(drawn_table, ax=axis, shrink=0.85)
fig.tight_layout()

### 4.2 Crossings, which a model cannot be wrong about

Where an artery passes over a vein the pixel is annotated as **both**, so a model calling it either
is right, and every score is flattered a little by however many of them there are. This says how
many that is, per dataset, on the same sample.

In [ ]:
crossings = []
for dataset, keys in CLASSED.items():
    for key in keys:
        artery, vein = drawn(dataset, key, "artery"), drawn(dataset, key, "vein")
        union = artery | vein
        crossings.append(
            {
                "dataset": dataset,
                "share of vessel pixels annotated as both": float((artery & vein).sum())
                / max(int(union.sum()), 1),
            }
        )
pd.DataFrame(crossings).groupby("dataset").mean().round(4)

## 5. Against the annotators

**No dataset built for this benchmark keeps its annotators apart.** Each publishes one artery/vein
map per photograph, by one reader or by a consensus that is not decomposable, so there is no
reader-against-reader band here of the kind the disc benchmark has — no ceiling to read the models
against, and no way to tell a model that is wrong where the experts would have disagreed from one
that is wrong where they would not.

What can be said is how independent the ground truths are of **each other**, which section 4 of the
build skill records per dataset and the run carries in `benchmark.DERIVED`.

In [ ]:
pd.DataFrame(
    [
        {
            "dataset": dataset,
            "annotators named": len(
                {r for r in str(manifest(dataset)["readers"].iloc[0]).split(";") if r and r != "nan"}
            ),
            "vessel annotation": benchmark.DERIVED.get(dataset, "unknown"),
        }
        for dataset in DATASETS
    ]
).set_index("dataset")

**Why that column matters.** In three of these four datasets the vessel annotation *is* the
artery/vein annotation — Fundus-AVSeg and AVRDB derive theirs outright, and HRF's hand-drawn gold
standard differs from the union of its artery/vein maps by 0.004% of pixels. So a model's vessel
score and its class scores there are **one measurement seen twice**, and their agreement is not
corroboration of anything.

### 5.1 How wide the annotator drew

Every model scores far lower on AVRDB than on the other three, and by a margin — a quarter of a
Dice — too large to be five different networks each failing on the same hundred photographs in its
own way. When every model agrees and all of them disagree with the annotation, the annotation is
what section 6 says is worth a second look.

Two things are measured here, on the same sample as section 4. **How much of the frame the
annotator's vessels cover**, beside how much each model's cover: a reference drawn wider than the
vessels costs overlap from every model at once, and costs it whether the model found the vessels or
not. And **the gap between clDice and Dice**, per dataset, which says whether what the models lost
was the network or only the width — the distinction section 3 exists to draw.

In [ ]:
width = []
for dataset, keys in sampled.items():
    for key in keys:
        annotated = float(drawn(dataset, key, "vessels").mean())
        for model in MODELS:
            try:
                width.append(
                    {
                        "dataset": dataset,
                        "key": key,
                        "model": model,
                        "annotated share of frame": annotated,
                        "model share of frame": float(said(model, dataset, key, "vessels").mean()),
                    }
                )
            except FileNotFoundError:
                continue
drawn_width = pd.DataFrame(width)
drawn_width["annotated ÷ model"] = (
    drawn_width["annotated share of frame"] / drawn_width["model share of frame"]
)

pd.DataFrame(
    {
        "annotated vessels, share of frame": drawn_width.groupby("dataset")[
            "annotated share of frame"
        ].mean(),
        "models' vessels, share of frame": drawn_width.groupby("dataset")[
            "model share of frame"
        ].mean(),
        "annotated ÷ model": drawn_width.groupby("dataset")["annotated ÷ model"].mean(),
        "mean clDice − Dice, every model": gap.groupby("dataset")["gap"].mean(),
    }
).round(3)

## 6. Model against model

Where two models agree, that is a fact about the models, not evidence that either is right —
especially here, where several trained on overlapping public data. The useful reading is the
opposite one: photographs where every model agrees with every other and all of them disagree with
the annotator are photographs where the annotation is worth a second look.

In [ ]:
wide = GRADED.pivot_table(index=["dataset", "key"], columns="model", values="vessels_dice")
wide.corr(method="spearman").round(3)

In [ ]:
consensus = wide.dropna()
print(f"{len(consensus):,} photographs every model scored")
hardest = consensus.mean(axis=1).sort_values()
print("\nphotographs every model found hard (mean vessels Dice):")
print(hardest.head(10).round(3).to_string())

### 6.1 Agreement between two models' masks, rather than between their scores

Two models can score alike on a photograph and have drawn different vessels. This compares the
masks themselves — Dice between one model's vessels and another's — on the same sample as section
4, using the run's own metric rather than a second implementation of it.

In [ ]:
agreement = pd.DataFrame(index=MODELS, columns=MODELS, dtype=float)
for first in MODELS:
    for second in MODELS:
        scores = []
        for dataset, keys in sampled.items():
            for key in keys:
                try:
                    scores.append(
                        measurements.dice(
                            said(first, dataset, key, "vessels"),
                            said(second, dataset, key, "vessels"),
                        )
                    )
                except FileNotFoundError:
                    continue
        kept = [value for value in scores if value is not None]
        agreement.loc[first, second] = float(np.mean(kept)) if kept else np.nan
agreement.round(3)

## 7. The hard cases

The photographs every model got wrong, drawn rather than tabulated, **taken from each dataset in
turn** — a grid drawn from whichever dataset happens to hold the most failures is a picture of that
dataset rather than of the benchmark. The annotator's vessels are in white; each model's are beside
them.

In [ ]:
def thumbnail(mask: np.ndarray, side: int = 320) -> np.ndarray:
    """A mask small enough to draw, without pretending the detail survived."""
    return np.asarray(Image.fromarray(mask).resize((side, side), Image.NEAREST))


PER_DATASET = 2
for dataset in DATASETS:
    mine = GRADED[GRADED["dataset"] == dataset]
    worst = (
        mine.groupby("key")["vessels_dice"].mean().sort_values().head(PER_DATASET).index.tolist()
    )
    if not worst:
        continue
    fig, axes = plt.subplots(
        len(worst), len(MODELS) + 2, figsize=(2.2 * (len(MODELS) + 2), 2.4 * len(worst))
    )
    axes = np.atleast_2d(axes)
    for row, key in enumerate(worst):
        with Image.open(STORE / dataset / "512" / "images" / f"{key}.png") as photograph:
            axes[row, 0].imshow(np.asarray(photograph.convert("RGB")))
        axes[row, 0].set_ylabel(key, fontsize=8)
        axes[row, 0].set_title("photograph" if row == 0 else "", fontsize=9)
        axes[row, 1].imshow(thumbnail(drawn(dataset, key, "vessels")), cmap="gray")
        axes[row, 1].set_title("annotator" if row == 0 else "", fontsize=9)
        for column, model in enumerate(MODELS, start=2):
            try:
                axes[row, column].imshow(
                    thumbnail(said(model, dataset, key, "vessels")), cmap="gray"
                )
            except FileNotFoundError:
                axes[row, column].text(0.5, 0.5, "failed", ha="center", va="center")
            score = mine.loc[(mine["key"] == key) & (mine["model"] == model), "vessels_dice"]
            axes[row, column].set_title(
                f"{model}\n{float(score.iloc[0]):.2f}" if len(score) else model, fontsize=8
            )
        for axis in axes[row]:
            axis.set_xticks([])
            axis.set_yticks([])
    fig.suptitle(f"{dataset} — the photographs every model found hardest", fontsize=11)
    fig.tight_layout()
    plt.show()

## 8. What each model costs

Seconds per photograph, as the run recorded them, **beside the device**. This measures the machine
as much as the model: the same weights on another processor give another number, and a model twice
as slow for a hundredth of Dice is a different proposition at fifty thousand photographs than at
fifty.

The figure covers the model's own call and nothing around it — not reading the photograph, not
scoring the answer — and leaves out the batch that loaded the weights.

In [ ]:
cost = overview.reset_index().pivot_table(
    index="model", columns="dataset", values="seconds_per_photograph"
)
cost["device"] = [
    overview.loc[model, "device"].dropna().iloc[0] if overview.loc[model, "device"].notna().any()
    else "—"
    for model in cost.index
]
cost.round(2)

In [ ]:
# The same seconds pooled two ways, because the two answer different questions and disagree for
# one model. Weighting by photograph says what a collection of these 804 costs; averaging the four
# datasets lets HRF's 45 photographs weigh as much as REYIA's 559, which is what the index page
# reports.
timing = overview.reset_index().dropna(subset=["seconds_per_photograph"])
timing["seconds"] = timing["seconds_per_photograph"] * timing["processed"]
pooled_cost = pd.DataFrame(
    {
        "per photograph": timing.groupby("model")["seconds"].sum()
        / timing.groupby("model")["processed"].sum(),
        "averaged over datasets": timing.groupby("model")["seconds_per_photograph"].mean(),
    }
)
pooled_cost["difference"] = pooled_cost["averaged over datasets"] - pooled_cost["per photograph"]
pooled_cost.sort_values("per photograph").round(2)

In [ ]:
fig, axis = plt.subplots(figsize=(9, 3.4))
speed = overview.groupby("model")["seconds_per_photograph"].mean()
quality = GRADED.groupby("model")["vessels_dice"].mean()
for model in MODELS:
    if model in speed and model in quality:
        axis.scatter(speed[model], quality[model], s=60)
        axis.annotate(model, (speed[model], quality[model]), fontsize=8,
                      xytext=(4, 4), textcoords="offset points")
axis.set_xlabel("seconds per photograph")
axis.set_ylabel("mean vessels Dice")
axis.set_title("What agreement costs in time")
axis.grid(alpha=0.3)
fig.tight_layout()

## 9. What this analysis cannot say

- **Contamination.** A model scored on a dataset it trained on is not being tested. The marks below
  come from each model's catalogue page, and `unknown` is never merged with `out-of-sample`: one
  says the model did not train on these images, the other that nobody established what it trained
  on.
- **The vessel score is not a second opinion.** In three of these datasets it is the artery/vein
  annotation measured again (section 5).
- **No ceiling.** No dataset here keeps its annotators apart, so there is no reader-against-reader
  band to read the models against, and a model at the top of a column may still be below what two
  experts would agree on.
- **Nothing about calibre, ratio or tortuosity.** Those are the biomarker benchmark's, computed
  from the masks this one kept under `.atlas_runs/av/`.
- **Resampling.** Every model works at a grid other than the annotator's, and its probabilities are
  carried to the native frame and thresholded there. A model whose grid is furthest from the
  photograph's own resolution is the one that rule flatters or punishes most, and that is not
  separated here.

In [ ]:
pd.DataFrame(
    {
        dataset: {model: contamination.mark(model, dataset) for model in MODELS}
        for dataset in DATASETS
    }
)

In [ ]:
resolutions = pd.DataFrame(
    [
        {
            "dataset": dataset,
            "photographs": len(manifest(dataset)),
            "native side, median": int(
                pd.to_numeric(manifest(dataset)["crop_side"]).median()
            ),
            "native side, range": (
                f"{int(pd.to_numeric(manifest(dataset)['crop_side']).min()):,}"
                f"–{int(pd.to_numeric(manifest(dataset)['crop_side']).max()):,}"
            ),
        }
        for dataset in DATASETS
    ]
).set_index("dataset")
resolutions